# English Tutor PoC

Grammar-correction agent for English, modeled on `GermanTutorPoC.ipynb`. Grounded in
**New Round-Up 5 – English Grammar Practice** (Evans & Dooley, Pearson), an
intermediate-level reference covering Present/Past/Future Forms, Infinitive/-ing/Participles,
Modal Verbs, The Passive, Conditionals/Wishes, Clauses, Reported Speech, Nouns/Articles,
the Causative Form, Adjectives/Adverbs/Comparisons, Demonstratives/Pronouns/Possessives/
Quantifiers, Prepositions, and Questions & Answers.

In [1]:
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIModel
from pydantic_ai.providers.openai import OpenAIProvider

## 1. Build a searchable reference from the New Round-Up 5 PDF

We extract the PDF once into overlapping text chunks and cache them to JSON
(`new_round_up_5_reference.json`) next to this notebook, so re-running the notebook
doesn't require re-parsing the 200+ page PDF every time.

In [ ]:
import json
from pathlib import Path

PDF_PATH = Path("data/New-Round-Up-5.pdf")
REFERENCE_PATH = Path("new_round_up_5_reference.json")

CHUNK_SIZE = 1200
CHUNK_OVERLAP = 200


def _clean_page(text: str) -> str:
    lines = [l for l in text.splitlines() if l.strip() and "hasanboy" not in l.lower()]
    return "\n".join(lines)


def build_reference(pdf_path: Path) -> dict:
    from pypdf import PdfReader

    reader = PdfReader(str(pdf_path))
    chunks = []
    for page_num, page in enumerate(reader.pages, start=1):
        text = _clean_page(page.extract_text() or "")
        if len(text) < 40:
            continue
        start = 0
        while start < len(text):
            end = start + CHUNK_SIZE
            chunks.append({"page": page_num, "text": text[start:end]})
            if end >= len(text):
                break
            start = end - CHUNK_OVERLAP
    return {
        "source": pdf_path.name,
        "title": "New Round-Up 5 - English Grammar Practice (Evans & Dooley, Pearson)",
        "level": "intermediate",
        "chunks": chunks,
    }


if REFERENCE_PATH.exists():
    reference = json.loads(REFERENCE_PATH.read_text())
else:
    reference = build_reference(PDF_PATH)
    REFERENCE_PATH.write_text(json.dumps(reference, ensure_ascii=False))

print(f"Loaded {len(reference['chunks'])} reference chunks from {reference['source']}")

Loaded 527 reference chunks from New-Round-Up-5.pdf


## 2. Lightweight keyword search over the reference

No embedding model required for the PoC — a keyword-overlap score over the cached
chunks is enough for the agent to ground its grammar explanations in the book's own
wording via the `grammar_reference_lookup` tool below.

In [3]:
import re
from collections import Counter

STOPWORDS = {"the", "a", "an", "of", "and", "in", "to", "for", "is", "are", "on", "with", "this", "that"}


def _tokenize(text: str) -> list[str]:
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w not in STOPWORDS and len(w) > 2]


_chunk_tokens = [Counter(_tokenize(c["text"])) for c in reference["chunks"]]


def search_reference(query: str, top_k: int = 3) -> str:
    """Keyword search over the New Round-Up 5 reference chunks."""
    q_tokens = _tokenize(query)
    scored = [
        (sum(tokens.get(t, 0) for t in q_tokens), i)
        for i, tokens in enumerate(_chunk_tokens)
    ]
    scored = [s for s in scored if s[0] > 0]
    scored.sort(reverse=True)
    top = scored[:top_k]
    if not top:
        return "No matching section found in New Round-Up 5 for this query."
    results = []
    for _, i in top:
        chunk = reference["chunks"][i]
        results.append(f"[p.{chunk['page']}] {chunk['text'].strip()}")
    return "\n\n---\n\n".join(results)


print(search_reference("present perfect vs past simple", top_k=1))

[p.112] onths / years, etc. before
• When the reporting verb is in the past, the verb tenses change as follows:
Direct speech Reported speech
present simple
“Tom needs a new bike," Dad said.
past simple
Dad said Tom needed a new bike.
present continuous
“He is watching TV," she said.
past continuous
She said he was watching TV.
present perfect
“We has just left, ” she said.
past perfect
She said he had just left.
past simple
“He left an hour ago," she said.
past simple or past perfect
She said he (had) left an hour before.
past continuous
7 was surfing the Net at two o'clock yesterday," 
he said.
past continuous or past perfect continuous 
He said he was surfing / had been surfing the
Net at two o'clock the day before.
future
“He 'll be back in an hour," she said.
conditional
She said he would be back in an hour.
present perfect continuous
"I've been typing since morning, ” she said.
past perfect continuous
She said she had been typing since morning.
If the direct verb is already in th

In [4]:
# Ollama exposes an OpenAI-compatible API — point directly at it
model = OpenAIModel(
    model_name="gemma4:26b",
    provider=OpenAIProvider(base_url="http://localhost:11434/v1"),
)

print("Model configured:", model)

Model configured: OpenAIModel()


/var/folders/6k/3yb_sy2d4plffrxc253561_40000gp/T/ipykernel_31725/896140090.py:2: DeprecationWarning: `OpenAIModel` was renamed to `OpenAIChatModel` to clearly distinguish it from `OpenAIResponsesModel` which uses OpenAI's newer Responses API. Use that unless you're using an OpenAI Chat Completions-compatible API, or require a feature that the Responses API doesn't support yet like audio.
  model = OpenAIModel(


In [ ]:
from pydantic import BaseModel


class TopicDetection(BaseModel):
    topics: list[str]


class Violation(BaseModel):
    rule: str
    explanation: str
    example_error: str
    example_correction: str
    count: int


class GrammarSummary(BaseModel):
    violations: list[Violation]
    confidence: float  # 0.0 - 1.0

In [ ]:
topic_agent = Agent(
    model=model,
    output_type=TopicDetection,
    retries=3,
    system_prompt=(
        "You are an English grammar checker. Read the learner's text and identify which grammar "
        "topics from New Round-Up 5 (Evans & Dooley, Pearson) it violates, choosing from: "
        "Present/Past/Future Forms, Infinitive/-ing form/Participles, Modal Verbs, The Passive, "
        "Conditionals/Wishes, Clauses, Reported Speech, Nouns/Articles, the Causative Form, "
        "Adjectives/Adverbs/Comparisons, Demonstratives/Pronouns/Possessives/Quantifiers, "
        "Prepositions, Questions & Answers. Return only the topic names — no explanations, no "
        "corrections. Return an empty list if the text has no grammar errors. You must always "
        "respond by calling the required output function — never reply with plain text, even "
        "when the list is empty."
    ),
)

explain_agent = Agent(
    model=model,
    output_type=GrammarSummary,
    retries=3,
    system_prompt=(
        "You are an English grammar tutor. You are given a learner's text and reference material "
        "from 'New Round-Up 5 English Grammar Practice' (Evans & Dooley, Pearson) already looked up "
        "for the topics this text touches — ground every explanation in that material. "
        "For each violated rule: explain it simply, give one example error/correction pair that is "
        "NOT taken from the learner's text, and report the violation count. Never say where in the "
        "text the errors occur — the learner must find and fix them by re-reading their own text. "
        "Each violation MUST be reported using exactly these JSON fields: 'rule', 'explanation', "
        "'example_error', 'example_correction', 'count' — do not use any other field names. "
        "You must always respond by calling the required output function with the GrammarSummary "
        "schema — never reply with plain text. If the text has no grammar errors, call the output "
        "function with an empty violations list and a confidence near 1.0."
    ),
)

In [ ]:
_agent_memory: list[dict] = []


@explain_agent.tool_plain
def memory_search(query: str) -> str:
    """Search the agent's memory of previously seen errors and rules."""
    q_tokens = set(_tokenize(query))
    hits = [
        m for m in _agent_memory
        if q_tokens & set(_tokenize(m["rule"] + " " + m["error"]))
    ]
    if not hits:
        return f"No memory entries match query: '{query}'"
    return "\n".join(f"- rule: {m['rule']!r}, error: {m['error']!r}" for m in hits)


@explain_agent.tool_plain
def store_errors_in_memory(errors: list[str]) -> str:
    """Store identified errors in the agent's memory. Each entry is 'rule: error'."""
    for e in errors:
        rule, _, error = e.partition(":")
        _agent_memory.append({"rule": rule.strip(), "error": error.strip() or rule.strip()})
    return f"Stored {len(errors)} error(s) in memory (total: {len(_agent_memory)})."

## 2b. Control flow lives in Python, not in prose

The system prompt used to ask the model, in English sentences, to enforce eight
behaviours: stop after 5 iterations on the same text, stop on empty input, stop on
`stop`/`exit`, stop if the input isn't English, stop if there are no errors, split
texts over 250 words, cluster repeated violations of the same rule, and always look
up the reference before explaining. None of that was verified — it was a request to
a 26B local model, not a control.

All eight are now plain Python below:
- `fetch_reference_for_topics` / `check_chunk` — the reference lookup always runs
  between topic detection and explanation; the model never decides whether to call it.
- `is_stop_keyword`, `is_english`, `split_into_chunks`, `cluster_violations` — pure
  functions, independently testable.
- `TutorSession` — tracks the iteration count per text in Python; the model has no
  memory of how many times it has seen a given text.

In [ ]:
def fetch_reference_for_topics(topics: list[str]) -> str:
    """Deterministically look up reference material for each detected topic — always runs,
    regardless of what the explanation model would have chosen to do."""
    if not topics:
        return "No grammar topics detected."
    sections = [f"### {topic}\n{search_reference(topic, top_k=2)}" for topic in topics]
    return "\n\n".join(sections)


async def check_chunk(chunk: str) -> GrammarSummary:
    """Detect violated topics, fetch grounding material for them, then explain — in that order,
    enforced by the call sequence rather than by asking the model nicely."""
    detection = (await topic_agent.run(chunk)).output
    reference_material = fetch_reference_for_topics(detection.topics)
    prompt = (
        f"Learner's text:\n{chunk}\n\n"
        f"Reference material (already looked up for the topics this text touches):\n{reference_material}"
    )
    result = await explain_agent.run(prompt)
    return result.output

In [ ]:
# pip install langdetect
from langdetect import LangDetectException, detect

MAX_ITERATIONS = 5
WORD_SPLIT_THRESHOLD = 250
STOP_KEYWORDS = {"stop", "exit"}


def is_stop_keyword(text: str) -> bool:
    return text.strip().lower() in STOP_KEYWORDS


def is_english(text: str) -> bool:
    try:
        return detect(text) == "en"
    except LangDetectException:
        return False


def split_into_chunks(text: str, max_words: int = WORD_SPLIT_THRESHOLD) -> list[str]:
    words = text.split()
    if not words:
        return []
    return [" ".join(words[i:i + max_words]) for i in range(0, len(words), max_words)]


def cluster_violations(violations: list[Violation]) -> list[Violation]:
    clustered: dict[str, Violation] = {}
    for v in violations:
        if v.rule in clustered:
            clustered[v.rule].count += v.count
        else:
            clustered[v.rule] = v.model_copy()
    return list(clustered.values())

In [ ]:
from typing import Optional, Union


class TutorSession:
    """Tracks how many times the same text has been submitted, so the 5-iteration
    cap is enforced by a counter instead of the model tracking it in conversation."""

    def __init__(self):
        self.text_key: Optional[str] = None
        self.iteration = 0

    def register(self, user_input: str) -> bool:
        """Returns False once the iteration cap for this exact text is reached."""
        if user_input != self.text_key:
            self.text_key = user_input
            self.iteration = 1
        else:
            self.iteration += 1
        return self.iteration <= MAX_ITERATIONS


async def check_text(session: TutorSession, user_input: str) -> Union[GrammarSummary, str]:
    if not user_input.strip():
        return "Empty input — nothing to check."
    if is_stop_keyword(user_input):
        return "Session stopped."
    if not is_english(user_input):
        return "Input is not English — please write in English."
    if not session.register(user_input):
        return f"Reached the {MAX_ITERATIONS}-iteration limit for this text."

    violations: list[Violation] = []
    confidences: list[float] = []
    for chunk in split_into_chunks(user_input):
        summary = await check_chunk(chunk)
        violations.extend(summary.violations)
        confidences.append(summary.confidence)

    clustered = cluster_violations(violations)
    if not clustered:
        return "No errors found — well done!"

    return GrammarSummary(violations=clustered, confidence=sum(confidences) / len(confidences))

## 3. First iteration — run the agent on flawed intermediate-level English

The sample text below deliberately exercises several New Round-Up 5 chapters: Present
Perfect vs Past Simple, Past Continuous, Modal Verbs (`must`), and Nouns/Articles.

In [ ]:
session = TutorSession()

sample_text = (
    "I have went to London last year and I seen a lot of interesting place. "
    "I must to visit the Big Ben because my friend told me it's beautiful monument. "
    "Yesterday, I am walking in park when it start raining."
)

output = await check_text(session, sample_text)

if isinstance(output, GrammarSummary):
    print(output.model_dump_json(indent=2))
else:
    print(output)

## 4. Iterative correction loop (multi-turn)

Mirrors the intended workflow: the learner submits a correction, the agent re-checks it
and the summary should shrink as fewer rules are violated.

In [ ]:
from typing import Union


def render(result: Union[GrammarSummary, str]) -> str:
    if isinstance(result, str):
        return result
    lines = [
        f"- {v.rule} ({v.count}x): {v.explanation}\n"
        f'  e.g. "{v.example_error}" -> "{v.example_correction}"'
        for v in result.violations
    ]
    return "\n".join(lines)


session = TutorSession()

# Turn 1: original flawed text
r1 = await check_text(session, sample_text)
print("Turn 1 summary:\n", render(r1), "\n")

# Turn 2: the learner's attempted correction (still has a couple of slips)
corrected_text = (
    "I went to London last year and I saw a lot of interesting places. "
    "I must visit Big Ben because my friend told me it is a beautiful monument. "
    "Yesterday, I was walking in the park when it started raining."
)
r2 = await check_text(session, corrected_text)
print("Turn 2 summary:\n", render(r2))